# 07 — Borg Transwarp Debugging

Two-panel chart for debugging the Borg Transwarp indicator:
- **Top**: SPX candlestick with DSTFS candle colors
- **Bottom**: QQQ allocation bars (green/cyan/red) + overbought/oversold markers

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Fetch Data

Connect to TastyTrade and fetch SPX candles (for the candlestick) + all 20 Borg tickers.

In [ ]:
import asyncio

from options_analyzer.config import load_config
from options_analyzer.engine.borg_transwarp import BORG_TICKERS
from options_analyzer.factory import create_providers

config = load_config()
providers = await create_providers(config)
market_data = providers.market_data
print(f"Connected to {providers.provider_name}.")

# SPX for the candlestick chart
spx_candles = await market_data.get_candles("SPX", interval="1d", days_back=180)
print(f"SPX: {len(spx_candles)} bars ({spx_candles.timestamps[0]:%Y-%m-%d} to {spx_candles.timestamps[-1]:%Y-%m-%d})")

# All 20 Borg tickers
tasks = {sym: market_data.get_candles(sym, interval="1d", days_back=180) for sym in BORG_TICKERS}
results = await asyncio.gather(*tasks.values(), return_exceptions=True)

candle_data = {}
for sym, res in zip(tasks.keys(), results):
    if isinstance(res, Exception):
        print(f"  WARN: {sym} failed — {res}")
    else:
        candle_data[sym] = res

print(f"Fetched {len(candle_data)}/{len(BORG_TICKERS)} Borg tickers")

## 2. Borg Transwarp Chart

In [ ]:
from options_analyzer.engine.borg_transwarp import compute_borg_transwarp_series
from options_analyzer.visualization.market_charts import plot_borg_candlestick

borg_closes = {sym: candle_data[sym].closes for sym in BORG_TICKERS if sym in candle_data}
borg_results = compute_borg_transwarp_series(borg_closes)
print(f"Borg series: {len(borg_results)} bars")

fig = plot_borg_candlestick(
    borg_results,
    opens=spx_candles.opens,
    highs=spx_candles.highs,
    lows=spx_candles.lows,
    closes=spx_candles.closes,
    timestamps=spx_candles.timestamps,
    title="SPX — Borg Transwarp Debugging (Daily)",
)
fig.update_layout(height=800)
fig.show()

## 3. Cleanup

In [ ]:
await providers.disconnect()
print("Session disconnected.")